In [34]:
import pandas as pd
import multiprocessing as mp
from tqdm import tqdm
from protocols import utils, functions

In [30]:
# the path to the CRyPTIC v3.1.0 data tables (including the DST_MEASUREMENTS_+.pkl which has DST for the validation samples appended)
cryptic_tables_path = '/Users/fowler/Dropbox/files/cryptic/cryptic-release-three/cryptic-tables-v3.1.0/'

# whether to run the processes which take a long time
run_long_processes = True

# number of cores
n_cores = 6

drug_genes = {
    'BDQ': {"genes": ["Rv0678", "atpE", "pepQ"], "phylogenetic": []},
    'CFZ': {"genes": ["Rv0678", "atpE", "pepQ"], "phylogenetic": []},
    "AMI": {"genes": ["eis", "rrs"], "phylogenetic": []},
    "CAP": {"genes": ["rrs", "tlyA"], "phylogenetic": []},
    "DLM": {"genes": ["ddn"], "phylogenetic": []},
    "EMB": {"genes": ["embA", "embB"], "phylogenetic": []},
    "ETH": {"genes": ["ethA", "fabG1", "inhA"], "phylogenetic": []},
    "INH": {"genes": ["katG", "inhA", "ahpC", "fabG1"], "phylogenetic": []},
    "KAN": {"genes": ["eis", "rrs"], "phylogenetic": []},
    "LEV": {"genes": ["gyrA", "gyrB"], "phylogenetic": ['gyrA@S95T']},
    "LZD": {"genes": ["rplC"], "phylogenetic": []},
    "MXF": {"genes": ["gyrA", "gyrB"], "phylogenetic": ['gyrA@S95T']},
    "RIF": {"genes": ["rpoB"], "phylogenetic": []},
    "STM": {"genes": ["gid", "rpsL", "rrs"], "phylogenetic": []},
    'PZA': {"genes": ["pncA"], "phylogenetic": []},    
}

who_drugs = list(pd.read_csv('data/who2_drugs.csv').drug)

In [32]:
def process_performance(args):
    drug, genes, catalogues, dataset, training, FRS = args
    results = []

    if drug in who_drugs:
        
        mutations = functions.prep_mutations(
            'data/mutations-v3.1.0/', 
            genes, 
            version='v3.1.0', 
            mut_path=cryptic_tables_path+'MUTATIONS.parquet', 
            var_path=cryptic_tables_path+'VARIANTS.parquet',
            train=training
        )
        if dataset == 'training':
            version = 'v1.0'
            validation = False
        elif dataset == 'all':
            version = 'v3.0'
            validation = False
        elif dataset == 'validation':    
            version = 'v3.0'
            validation = True
        else:
            raise ValueError('dataset must be one of "training", "all", or "validation"')

        phenotypes = functions.prep_phenotypes(
            drug,
            cryptic_tables_path+'DST_MEASUREMENTS_+.pkl',
            cryptic_tables_path+'GENOMES.parquet',
            cryptic_tables_path+'WGS_SAMPLES.parquet',
            version,
            validation=validation
        )
        phenotypes.set_index('UNIQUEID', inplace=True)
        mutations.set_index('UNIQUEID', inplace=True)
        all_data = phenotypes.join(mutations[mutations.FRS >= FRS], how='left')
        all_data.reset_index(inplace=True)

        if len(all_data)>0:

            for cat_name in catalogues:

                _, cov, sens, spec, sens2, spec2 = utils.piezo_predict(iso_df=all_data, drug=drug, catalogue_file=catalogues[cat_name])

                results.append({
                    'DRUG': drug,
                    'catalogue': cat_name,
                    'SENSITIVITY': sens,
                    'SPECIFICITY': spec,
                    'COVERAGE': cov,
                    'SENSITIVITY2': sens2,
                    'SPECIFICITY2': spec2,
                })

    return results

def parallel_performance_evaluation(drug_genes, catalogues, dataset, training, frs):
    tasks = [(drug, data['genes'], catalogues, dataset, training, frs) for drug, data in drug_genes.items()]
    
    # Works on Mac/Linux
    ctx = mp.get_context("fork") 
    
    # Don't use more workers than tasks
    num_workers = min(n_cores, len(tasks))  

    # Run in parallel
    with ctx.Pool(num_workers) as pool:
        all_results = list(tqdm(pool.imap(process_performance, tasks), total=len(tasks)))
    
    results_df = pd.DataFrame([item for sublist in all_results for item in sublist])

    return results_df



In [36]:
if run_long_processes:

    catalogues = {
        'WHOv1': 'catalogues/whov1/NC_000962.3_WHO-UCN-GTB-PCI-2021.7_v1.2_GARC1_RUS.csv',
        'WHOv2': 'catalogues/whov2/NC_000962.3_WHO-UCN-TB-2023.5_v2.0_GARC1_RFUS.csv',
        'catomatic_v1': "catalogues/catomatic_v1.csv",
    }

    results_all = parallel_performance_evaluation(drug_genes, catalogues, dataset='training', training=False, frs=0.1)
    results_all.to_csv('results/performance/whov1_whov2_cat1_training.csv')

  0%|          | 0/15 [00:00<?, ?it/s]

100%|██████████| 15/15 [17:42<00:00, 70.82s/it]


In [35]:
if run_long_processes:

    catalogues = {
        'WHOv1': 'catalogues/whov1/NC_000962.3_WHO-UCN-GTB-PCI-2021.7_v1.2_GARC1_RUS.csv',
        'WHOv2': 'catalogues/whov2/NC_000962.3_WHO-UCN-TB-2023.5_v2.0_GARC1_RFUS.csv',
        'catomatic_v1': "catalogues/catomatic_v1.csv",
    }

    results_all = parallel_performance_evaluation(drug_genes, catalogues, dataset='validation', training=False, frs=0.1)
    results_all.to_csv('results/performance/whov1_whov2_cat1_validation.csv')

100%|██████████| 15/15 [01:16<00:00,  5.11s/it]
